
# 🚨 Live Crypto Momentum Scanner + Slack Alerts

Continuously scans the Top-100 crypto universe using the same factor engine,
composite scoring, and signal state machine as
`crypto_ranked_asset_allocation_v3` — and pushes **Slack alerts** on state
transitions (ENTRY / WEAKENING / EXIT) and rank-acceleration surges. You
trade manually off the alerts; this notebook places no orders.

## Honest framing: a notebook is not where a "live engine" belongs

A cell with `while True` blocks the kernel for as long as it runs. It works —
Jupyter will happily execute it for hours and stream output/alerts live — but
it has real limitations you should know before you rely on it:

| Limitation | Consequence |
|---|---|
| Kernel dies -> loop dies | Laptop sleep, browser tab close (on hosted Jupyter), kernel restart, or a crash silently stops all scanning and all alerts. |
| No process supervisor | Nothing restarts it automatically. Compare to `systemd`'s `Restart=on-failure`, which the standalone-script version of this gets for free. |
| Single point of failure | One unhandled exception outside the loop's own try/except kills the whole thing. |

**If you keep this running unattended for real trading decisions**, the
heartbeat cell near the end is your canary — if Slack goes quiet for longer
than `heartbeat_interval_minutes`, assume the kernel died and go check, don't
assume no news is good news.

## Setup

```
pip install ccxt requests numpy pandas
```

Set your Slack webhook URL in the config cell below (or as an environment
variable `SLACK_WEBHOOK_URL` before launching Jupyter). Get one from:
Slack -> your workspace -> Apps -> Incoming Webhooks -> Add New Webhook.


In [ ]:

import os
import sys
import time
import sqlite3
import uuid
import logging
from datetime import datetime, timezone

import requests
import numpy as np
import pandas as pd
import ccxt

from IPython.display import display


In [ ]:

# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    # Universe
    "top_market_cap": 100,
    "quote_currency": "USDT",
    "min_24h_volume_usd": 5_000_000,

    # OHLCV
    "ohlcv_limit": 90,
    "timeframe": "1h",  # intraday cadence -- daily bars are too slow for "live"

    # Momentum windows (in bars, i.e. hours at timeframe="1h")
    "momentum_windows": [1, 4, 12, 24],
    "momentum_weights": [0.10, 0.20, 0.30, 0.40],

    # Relative strength
    "relative_strength_benchmark": "BTC/USDT",
    "relative_strength_window": 12,

    # Volume
    "volume_window": 20,
    "volume_accel_lookback": 5,

    # Technical confirmation
    "ema_fast": 20,
    "ema_slow": 50,

    # Risk / noise
    "volatility_window": 30,
    "correlation_window": 60,
    "entropy_window": 60,

    "min_history_bars": 65,

    # Same composite as crypto_ranked_asset_allocation_v3 -- carried over
    # deliberately rather than re-tuned here. Re-weighting should happen
    # against that notebook's robustness-verdict table, not ad hoc here.
    "factor_weights": {
        "momentum": 0.20,
        "relative_strength": 0.15,
        "volume_accel": 0.10,
        "volatility": 0.15,
        "correlation": 0.10,
        "entropy": 0.10,
        "volume": 0.10,
        "trend": 0.10,
    },

    # Signal state machine (same semantics as v3 notebook)
    "state": {
        "setup_score": 0.50,
        "entry_score": 0.65,
        "weakening_score_drop": 0.10,
        "exit_score": 0.45,
    },

    # Rank acceleration alert
    "rank_surge_threshold": 10,  # rank improved by >= this many positions since last scan

    # Loop
    "scan_interval_seconds": 300,        # 5 min between scans
    "universe_refresh_seconds": 1800,    # re-pull CoinGecko Top 100 every 30 min
    "max_consecutive_errors": 5,
    "error_backoff_seconds": 60,

    # Alerting
    "slack_webhook_url": os.environ.get("SLACK_WEBHOOK_URL", "PASTE_YOUR_SLACK_WEBHOOK_URL_HERE"),
    "alert_cooldown_minutes": 60,        # don't repeat the same (symbol, alert_type) within this window
    "heartbeat_interval_minutes": 360,   # "I'm alive" ping so silence doesn't mean "it crashed silently"

    # Storage
    "db_path": "live_scanner.db",
}

assert abs(sum(CONFIG["factor_weights"].values()) - 1.0) < 1e-9, "factor_weights must sum to 1.0"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler("live_scanner.log")],
)
logger = logging.getLogger("live_scanner")



## 🗄️ Persistence — same schema as v3, plus `alert_log`

`alert_log` is new: it's what lets `should_alert()` dedup/cooldown so you
don't get pinged every 5 minutes for a symbol sitting in `ENTRY`.


In [ ]:

# ============================================================
# SQLITE SCHEMA + PERSISTENCE
# ============================================================

def get_conn():
    conn = sqlite3.connect(CONFIG["db_path"])
    conn.execute("PRAGMA journal_mode=WAL;")
    return conn


def init_db():
    conn = get_conn()
    cur = conn.cursor()

    cur.execute('''
    CREATE TABLE IF NOT EXISTS scanner_history (
        run_id TEXT NOT NULL,
        run_timestamp TEXT NOT NULL,
        symbol TEXT NOT NULL,
        rank INTEGER,
        composite_score REAL,
        momentum REAL,
        relative_strength REAL,
        volume_accel REAL,
        volatility REAL,
        correlation REAL,
        entropy REAL,
        volume_ratio REAL,
        trend INTEGER,
        price REAL,
        PRIMARY KEY (run_id, symbol)
    );
    ''')

    cur.execute('''
    CREATE TABLE IF NOT EXISTS signal_state (
        symbol TEXT PRIMARY KEY,
        state TEXT NOT NULL,
        state_since TEXT NOT NULL,
        peak_score_while_held REAL,
        last_run_timestamp TEXT
    );
    ''')

    cur.execute('''
    CREATE TABLE IF NOT EXISTS state_transitions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT NOT NULL,
        from_state TEXT,
        to_state TEXT NOT NULL,
        run_timestamp TEXT NOT NULL,
        composite_score REAL,
        rank INTEGER
    );
    ''')

    cur.execute('''
    CREATE TABLE IF NOT EXISTS alert_log (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        symbol TEXT NOT NULL,
        alert_type TEXT NOT NULL,
        message TEXT,
        sent_at TEXT NOT NULL,
        composite_score REAL,
        rank INTEGER
    );
    ''')

    conn.commit()
    conn.close()


def save_snapshot(scores, run_timestamp):
    conn = get_conn()
    run_id = str(uuid.uuid4())
    out = scores.copy()
    out["run_id"] = run_id
    out["run_timestamp"] = run_timestamp
    cols = [
        "run_id", "run_timestamp", "symbol", "rank", "composite_score",
        "momentum", "relative_strength", "volume_accel", "volatility",
        "correlation", "entropy", "volume_ratio", "trend", "price",
    ]
    out[cols].to_sql("scanner_history", conn, if_exists="append", index=False)
    conn.close()
    return run_id


def load_previous_snapshot(before_timestamp):
    conn = get_conn()
    q = '''
        SELECT * FROM scanner_history
        WHERE run_timestamp = (
            SELECT MAX(run_timestamp) FROM scanner_history
            WHERE run_timestamp < ?
        )
    '''
    df = pd.read_sql_query(q, conn, params=(before_timestamp,))
    conn.close()
    return df


def load_states():
    conn = get_conn()
    df = pd.read_sql_query("SELECT * FROM signal_state", conn)
    conn.close()
    return df.set_index("symbol") if not df.empty else df


def last_alert_time(symbol, alert_type):
    conn = get_conn()
    q = "SELECT MAX(sent_at) as sent_at FROM alert_log WHERE symbol = ? AND alert_type = ?"
    row = pd.read_sql_query(q, conn, params=(symbol, alert_type))
    conn.close()
    val = row.iloc[0]["sent_at"]
    return pd.Timestamp(val, tz="UTC") if val else None


def log_alert(symbol, alert_type, message, sent_at, composite_score, rank):
    conn = get_conn()
    conn.execute(
        '''INSERT INTO alert_log (symbol, alert_type, message, sent_at, composite_score, rank)
           VALUES (?, ?, ?, ?, ?, ?)''',
        (symbol, alert_type, message, sent_at, composite_score, rank),
    )
    conn.commit()
    conn.close()


init_db()
print("DB ready:", CONFIG["db_path"])



## 📡 Data providers — exchange connection + universe + OHLCV


In [ ]:

# ============================================================
# DATA PROVIDERS
# ============================================================

COINGECKO_URL = "https://api.coingecko.com/api/v3/coins/markets"


def get_top_100_coins():
    params = {
        "vs_currency": "usd", "order": "market_cap_desc",
        "per_page": CONFIG["top_market_cap"], "page": 1, "sparkline": "false",
    }
    r = requests.get(COINGECKO_URL, params=params, timeout=20)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    cols = ["id", "symbol", "name", "market_cap_rank", "current_price",
            "market_cap", "total_volume", "price_change_percentage_24h"]
    return df[cols].copy()


def build_universe(exchange):
    top100 = get_top_100_coins()
    markets = exchange.load_markets()
    usdt_symbols = {
        m["base"].upper(): symbol
        for symbol, m in markets.items()
        if m.get("spot") and m.get("quote") == CONFIG["quote_currency"] and m.get("active", True)
    }
    top100["BASE"] = top100["symbol"].str.upper()
    universe = top100[
        (top100["total_volume"] >= CONFIG["min_24h_volume_usd"])
        & (top100["BASE"].isin(usdt_symbols))
    ].copy()
    universe["symbol"] = universe["BASE"].map(usdt_symbols)
    logger.info(f"Universe refreshed: {len(universe)} tradable+liquid of {len(top100)} top-100")
    return universe


def fetch_ohlcv(exchange, symbol, timeframe, limit):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
        df = pd.DataFrame(raw, columns=["timestamp", "open", "high", "low", "close", "volume"])
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
        return df.set_index("timestamp")
    except Exception as e:
        logger.warning(f"OHLCV fetch failed for {symbol}: {str(e)[:150]}")
        return None


def fetch_price_data(exchange, universe):
    price_data = {}
    for symbol in universe["symbol"]:
        df = fetch_ohlcv(exchange, symbol, CONFIG["timeframe"], CONFIG["ohlcv_limit"])
        if df is not None and len(df) >= CONFIG["min_history_bars"]:
            price_data[symbol] = df
        time.sleep(exchange.rateLimit / 1000)

    bench_symbol = CONFIG["relative_strength_benchmark"]
    if bench_symbol not in price_data:
        bench_df = fetch_ohlcv(exchange, bench_symbol, CONFIG["timeframe"], CONFIG["ohlcv_limit"])
        if bench_df is not None:
            price_data[bench_symbol] = bench_df

    return price_data


exchange = ccxt.binance({"enableRateLimit": True})
print("Exchange connected:", exchange.id)



## 🔧 Factor engine + cross-sectional scoring

Identical math to `crypto_ranked_asset_allocation_v3` — momentum, relative
strength, volume acceleration, volatility, correlation, entropy, trend —
reduced to intraday windows (`timeframe="1h"`, momentum windows in hours
instead of days) since this is meant to alert same-day, not next-week.


In [ ]:

# ============================================================
# FACTOR ENGINE
# ============================================================

def momentum_score_series(close):
    windows = CONFIG["momentum_windows"]
    weights = CONFIG["momentum_weights"]
    return sum(close.pct_change(w) * weight for w, weight in zip(windows, weights))


def relative_strength_series(close, benchmark_close, window):
    asset_ret = close.pct_change(window)
    bench_ret = benchmark_close.pct_change(window).reindex(close.index).ffill()
    return asset_ret - bench_ret


def realized_volatility(close, window):
    return close.pct_change().rolling(window).std() * np.sqrt(365)


def binary_entropy(close, window):
    r = close.pct_change()
    p = (r > 0).rolling(window).mean().clip(1e-12, 1 - 1e-12)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def volume_ratio_series(volume, window):
    return volume / volume.rolling(window).mean()


def volume_acceleration_series(volume, window, lookback):
    vr = volume_ratio_series(volume, window)
    return vr - vr.shift(lookback)


def trend_flag_series(close, fast, slow):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    return (ema_fast > ema_slow).astype(int)


def avg_abs_correlation(returns_matrix, symbol, window):
    sub = returns_matrix.tail(window)
    if symbol not in sub.columns or sub[symbol].dropna().shape[0] < window * 0.6:
        return np.nan
    corrs = sub.corr()[symbol].drop(labels=[symbol], errors="ignore")
    return corrs.abs().mean()


def build_factor_snapshot(price_data):
    close_matrix = pd.DataFrame({s: df["close"] for s, df in price_data.items()})
    returns_matrix = close_matrix.pct_change()

    bench_symbol = CONFIG["relative_strength_benchmark"]
    bench_close = price_data.get(bench_symbol, {}).get("close") if bench_symbol in price_data else None

    rows = []
    for symbol, df in price_data.items():
        close, volume = df["close"], df["volume"]
        if len(close) < CONFIG["min_history_bars"]:
            continue

        vol_ratio = volume_ratio_series(volume, CONFIG["volume_window"])
        vol_accel = volume_acceleration_series(volume, CONFIG["volume_window"], CONFIG["volume_accel_lookback"])
        trend = trend_flag_series(close, CONFIG["ema_fast"], CONFIG["ema_slow"])

        rows.append({
            "symbol": symbol,
            "timestamp": close.index[-1],
            "price": close.iloc[-1],
            "momentum": momentum_score_series(close).iloc[-1],
            "relative_strength": (
                relative_strength_series(close, bench_close, CONFIG["relative_strength_window"]).iloc[-1]
                if bench_close is not None and symbol != bench_symbol else 0.0
            ),
            "volatility": realized_volatility(close, CONFIG["volatility_window"]).iloc[-1],
            "entropy": binary_entropy(close, CONFIG["entropy_window"]).iloc[-1],
            "volume_ratio": vol_ratio.iloc[-1],
            "volume_accel": vol_accel.iloc[-1],
            "trend": trend.iloc[-1],
            "correlation": avg_abs_correlation(returns_matrix, symbol, CONFIG["correlation_window"]),
        })

    raw = pd.DataFrame(rows).dropna(subset=["momentum", "volatility", "entropy", "volume_ratio", "correlation"])
    return raw


def percentile_rank(series, higher_is_better=True):
    r = series.rank(pct=True, method="average")
    return r if higher_is_better else 1 - r


def build_scores(raw):
    df = raw.copy()
    df["momentum_score"] = percentile_rank(df["momentum"], True)
    df["relative_strength_score"] = percentile_rank(df["relative_strength"], True)
    df["volume_accel_score"] = percentile_rank(df["volume_accel"], True)
    df["volatility_score"] = percentile_rank(df["volatility"], False)
    df["correlation_score"] = percentile_rank(df["correlation"], False)
    df["entropy_score"] = percentile_rank(df["entropy"], False)
    df["volume_score"] = percentile_rank(df["volume_ratio"], True)
    df["trend_score"] = df["trend"].astype(float)

    w = CONFIG["factor_weights"]
    df["composite_score"] = (
        df["momentum_score"] * w["momentum"]
        + df["relative_strength_score"] * w["relative_strength"]
        + df["volume_accel_score"] * w["volume_accel"]
        + df["volatility_score"] * w["volatility"]
        + df["correlation_score"] * w["correlation"]
        + df["entropy_score"] * w["entropy"]
        + df["volume_score"] * w["volume"]
        + df["trend_score"] * w["trend"]
    )

    df = df.sort_values("composite_score", ascending=False).reset_index(drop=True)
    df["rank"] = df.index + 1
    return df



## 🚦 Signal state machine

Same `WATCH → SETUP → ENTRY → HOLD → WEAKENING → EXIT` machine as v3,
persisted per symbol in `signal_state`. Alerts fire on `ENTRY`, `WEAKENING`,
and `EXIT` — `SETUP`/`WATCH`/`HOLD` transitions are logged but not pushed to
Slack, so you're not alerted every cycle for a position sitting quietly in
`HOLD`.


In [ ]:

# ============================================================
# SIGNAL STATE MACHINE
# ============================================================

def transition(current_state, row, peak_score):
    s_cfg = CONFIG["state"]
    score = row["composite_score"]
    trend_ok = row["trend"] == 1
    volume_ok = row["volume_ratio"] > 1.0

    if current_state in (None, "EXIT"):
        return "WATCH", np.nan

    if current_state == "WATCH":
        return ("SETUP", np.nan) if score >= s_cfg["setup_score"] else ("WATCH", np.nan)

    if current_state == "SETUP":
        if score >= s_cfg["entry_score"] and trend_ok and volume_ok:
            return "ENTRY", score
        if score < s_cfg["setup_score"]:
            return "WATCH", np.nan
        return "SETUP", np.nan

    if current_state == "ENTRY":
        if score < s_cfg["exit_score"]:
            return "EXIT", np.nan
        return "HOLD", max(peak_score if pd.notna(peak_score) else score, score)

    if current_state in ("HOLD", "WEAKENING"):
        new_peak = max(peak_score if pd.notna(peak_score) else score, score)
        if score < s_cfg["exit_score"]:
            return "EXIT", np.nan
        if new_peak - score >= s_cfg["weakening_score_drop"]:
            return "WEAKENING", new_peak
        return "HOLD", new_peak

    return "WATCH", np.nan


def run_state_machine(scores, run_timestamp):
    existing = load_states()
    transitions_log = []
    new_rows = []

    for _, row in scores.iterrows():
        symbol = row["symbol"]
        if not existing.empty and symbol in existing.index:
            cur_state = existing.loc[symbol, "state"]
            peak = existing.loc[symbol, "peak_score_while_held"]
        else:
            cur_state, peak = None, np.nan

        new_state, new_peak = transition(cur_state, row, peak)

        if new_state != cur_state:
            transitions_log.append({
                "symbol": symbol, "from_state": cur_state, "to_state": new_state,
                "run_timestamp": run_timestamp, "composite_score": row["composite_score"],
                "rank": row["rank"],
            })

        new_rows.append({
            "symbol": symbol, "state": new_state,
            "state_since": run_timestamp if new_state != cur_state else (
                existing.loc[symbol, "state_since"] if (not existing.empty and symbol in existing.index) else run_timestamp
            ),
            "peak_score_while_held": new_peak, "last_run_timestamp": run_timestamp,
        })

    conn = get_conn()
    cur = conn.cursor()
    for r in new_rows:
        cur.execute('''
            INSERT INTO signal_state (symbol, state, state_since, peak_score_while_held, last_run_timestamp)
            VALUES (:symbol, :state, :state_since, :peak_score_while_held, :last_run_timestamp)
            ON CONFLICT(symbol) DO UPDATE SET
                state=excluded.state, state_since=excluded.state_since,
                peak_score_while_held=excluded.peak_score_while_held,
                last_run_timestamp=excluded.last_run_timestamp;
        ''', r)

    if transitions_log:
        pd.DataFrame(transitions_log).to_sql("state_transitions", conn, if_exists="append", index=False)

    conn.commit()
    conn.close()
    return pd.DataFrame(new_rows), pd.DataFrame(transitions_log)


def calculate_rank_change(current_scores, run_timestamp):
    previous = load_previous_snapshot(run_timestamp)
    x = current_scores[["symbol", "rank", "composite_score"]].copy()
    if previous.empty:
        x["previous_rank"], x["rank_change"] = np.nan, np.nan
        return x
    p = previous[["symbol", "rank"]].rename(columns={"rank": "previous_rank"})
    x = x.merge(p, on="symbol", how="left")
    x["rank_change"] = x["previous_rank"] - x["rank"]
    return x



## 💬 Slack alerting

Two alert families:

- **State-machine transitions** — `ENTRY` (new momentum trade confirmed by
  score + trend + volume), `WEAKENING` (score dropped meaningfully off its
  peak while held — consider trimming), `EXIT` (score floor breached).
- **Rank-acceleration surges** — a symbol jumping `rank_surge_threshold`+
  places since the last scan, *before* the state machine has confirmed
  `ENTRY`. This is the "which coin is starting to move fastest" signal from
  the original scanner design, surfaced as its own early-warning alert
  instead of only being buried inside the composite score.

Every alert is deduped against `alert_log` with a cooldown, so a symbol
sitting in `ENTRY` for six straight cycles doesn't spam you every 5 minutes.


In [ ]:

# ============================================================
# SLACK ALERTING
# ============================================================

STATE_EMOJI = {
    "ENTRY": "\U0001F525", "HOLD": "\u2705", "WEAKENING": "\u26A0\uFE0F",
    "EXIT": "\U0001F6AA", "SETUP": "\U0001F440", "WATCH": "\U0001F441\uFE0F",
}


def send_slack_message(text):
    webhook = CONFIG["slack_webhook_url"]
    if not webhook or webhook.startswith("PASTE_"):
        logger.warning("SLACK_WEBHOOK_URL not set -- alert not sent, logging only: " + text)
        return False
    try:
        resp = requests.post(webhook, json={"text": text}, timeout=10)
        resp.raise_for_status()
        return True
    except Exception as e:
        logger.error(f"Slack delivery failed: {str(e)[:200]}")
        return False


def should_alert(symbol, alert_type, cooldown_minutes):
    last = last_alert_time(symbol, alert_type)
    if last is None:
        return True
    return (datetime.now(timezone.utc) - last).total_seconds() >= cooldown_minutes * 60


def process_alerts(transitions_now, rank_accel, scores, run_timestamp_iso):
    scores_by_symbol = scores.set_index("symbol")
    cooldown = CONFIG["alert_cooldown_minutes"]

    alertable_states = {"ENTRY", "WEAKENING", "EXIT"}
    for _, t in transitions_now.iterrows():
        if t["to_state"] not in alertable_states:
            continue
        if not should_alert(t["symbol"], t["to_state"], cooldown):
            continue

        emoji = STATE_EMOJI.get(t["to_state"], "")
        row = scores_by_symbol.loc[t["symbol"]] if t["symbol"] in scores_by_symbol.index else None
        price = f"{row['price']:.4g}" if row is not None else "n/a"

        msg = (
            f"{emoji} *{t['symbol']}* -> *{t['to_state']}*\n"
            f"score: {t['composite_score']:.3f}  |  rank: #{int(t['rank'])}  |  price: {price}"
        )
        if send_slack_message(msg):
            log_alert(t["symbol"], t["to_state"], msg, run_timestamp_iso, t["composite_score"], t["rank"])

    surges = rank_accel[
        rank_accel["rank_change"].notna()
        & (rank_accel["rank_change"] >= CONFIG["rank_surge_threshold"])
    ]
    for _, r in surges.iterrows():
        if not should_alert(r["symbol"], "RANK_SURGE", cooldown):
            continue
        msg = (
            f"\U0001F680 *{r['symbol']}* rank surge: "
            f"#{int(r['previous_rank'])} -> #{int(r['rank'])} "
            f"(+{int(r['rank_change'])})  |  score: {r['composite_score']:.3f}"
        )
        if send_slack_message(msg):
            log_alert(r["symbol"], "RANK_SURGE", msg, run_timestamp_iso, r["composite_score"], r["rank"])


_last_heartbeat = {"ts": None}


def maybe_send_heartbeat(n_symbols):
    now = datetime.now(timezone.utc)
    last = _last_heartbeat["ts"]
    if last is not None and (now - last).total_seconds() < CONFIG["heartbeat_interval_minutes"] * 60:
        return
    send_slack_message(f"\U0001F49A Scanner heartbeat -- alive, scanning {n_symbols} symbols. {now.isoformat()}")
    _last_heartbeat["ts"] = now



## ▶️ Single-cycle test run

Run this cell once by itself before turning on the continuous loop below.
It exercises the full pipeline exactly once (universe → OHLCV → factors →
scoring → state machine → alerts) so you can see the output, confirm Slack
delivery, and check `scores`/`transitions_now` before letting anything run
unattended.


In [ ]:

# ============================================================
# ONE-OFF TEST CYCLE
# ============================================================

def run_scan_cycle(exchange, universe_cache):
    now = time.time()

    if universe_cache["universe"] is None or (now - universe_cache["last_refresh"]) >= CONFIG["universe_refresh_seconds"]:
        universe_cache["universe"] = build_universe(exchange)
        universe_cache["last_refresh"] = now

    universe = universe_cache["universe"]
    price_data = fetch_price_data(exchange, universe)

    if len(price_data) < 8:
        logger.warning(f"Only {len(price_data)} symbols with usable history this cycle -- skipping scoring.")
        return None, None, None

    raw = build_factor_snapshot(price_data)
    if raw.empty:
        logger.warning("Empty factor snapshot this cycle -- skipping.")
        return None, None, None

    scores = build_scores(raw)
    run_timestamp = datetime.now(timezone.utc).isoformat()

    save_snapshot(scores, run_timestamp)
    rank_accel = calculate_rank_change(scores, run_timestamp)
    _, transitions_now = run_state_machine(scores, run_timestamp)

    process_alerts(transitions_now, rank_accel, scores, run_timestamp)
    maybe_send_heartbeat(len(scores))

    return scores, rank_accel, transitions_now


universe_cache = {"universe": None, "last_refresh": 0}

scores, rank_accel, transitions_now = run_scan_cycle(exchange, universe_cache)

if scores is not None:
    print(f"Scored {len(scores)} symbols. Top 10:")
    display(scores[["rank", "symbol", "composite_score", "trend", "volume_ratio"]].head(10))

    if not transitions_now.empty:
        print("\nState transitions this cycle:")
        display(transitions_now)
    else:
        print("\nNo state transitions this cycle.")



## 🔁 Continuous live loop

Run this cell to scan every `CONFIG["scan_interval_seconds"]` until you stop
it. **Interrupt the kernel (Kernel → Interrupt, or the stop button) to end
it cleanly** — that's caught below and sends a "scanner stopped" message to
Slack so you know it went offline on purpose rather than crashing silently.

Leave this cell running in its own notebook / kernel if you're doing
anything else in Jupyter at the same time — a busy kernel can't run other
cells until you interrupt this one.


In [ ]:

# ============================================================
# CONTINUOUS LIVE LOOP -- run until interrupted (Kernel -> Interrupt)
# ============================================================

def run_live(max_cycles=None):
    consecutive_errors = 0
    cycle_count = 0

    logger.info("Live momentum scanner starting.")
    send_slack_message("\U0001F7E2 Live momentum scanner started.")

    try:
        while max_cycles is None or cycle_count < max_cycles:
            cycle_start = time.time()
            try:
                run_scan_cycle(exchange, universe_cache)
                consecutive_errors = 0
            except Exception as e:
                consecutive_errors += 1
                logger.exception(f"Scan cycle failed ({consecutive_errors}/{CONFIG['max_consecutive_errors']}): {e}")

                if consecutive_errors >= CONFIG["max_consecutive_errors"]:
                    send_slack_message(
                        f"\U0001F534 Scanner has failed {consecutive_errors} cycles in a row -- "
                        f"last error: {str(e)[:200]}. Backing off; check `live_scanner.log`."
                    )
                    time.sleep(CONFIG["error_backoff_seconds"] * consecutive_errors)
                else:
                    time.sleep(CONFIG["error_backoff_seconds"])

                cycle_count += 1
                continue

            cycle_count += 1
            elapsed = time.time() - cycle_start
            sleep_for = max(CONFIG["scan_interval_seconds"] - elapsed, 5)
            time.sleep(sleep_for)

    except KeyboardInterrupt:
        logger.info("Interrupted by user (Kernel -> Interrupt).")
    finally:
        send_slack_message("\U0001F7E1 Live momentum scanner stopped.")
        logger.info("Shutdown complete.")


# Example: run_live(max_cycles=3)   <- test with a bounded number of cycles first
# Uncomment the line below to run indefinitely until you interrupt the kernel:

# run_live()



## 📜 Review recent alerts / signal state

Quick queries against the same DB — useful after leaving `run_live()`
running for a while, or the morning after an overnight session.


In [ ]:

# ============================================================
# QUICK REVIEW QUERIES
# ============================================================

def recent_alerts(limit=25):
    conn = get_conn()
    df = pd.read_sql_query(
        "SELECT * FROM alert_log ORDER BY sent_at DESC LIMIT ?", conn, params=(limit,)
    )
    conn.close()
    return df


def current_signal_states():
    conn = get_conn()
    df = pd.read_sql_query(
        "SELECT * FROM signal_state ORDER BY last_run_timestamp DESC", conn
    )
    conn.close()
    return df


display(recent_alerts())
display(current_signal_states())



## ⚠️ Known limitations

1. **Kernel-bound liveness** (repeated from the top, because it's the one
   that actually bites people): no supervisor restarts this if it dies. If
   you need real unattended uptime — overnight, while you're at your day
   job — this belongs in a systemd/Docker-supervised script, not a notebook
   kernel, even though the code is identical either way.
2. **CoinGecko free tier rate limits.** `universe_refresh_seconds=1800` is
   set conservatively for that reason — pulling Top-100 every cycle instead
   of every 30 min will get you rate-limited.
3. **Alert cadence vs. your actual reaction time.** `scan_interval_seconds=300`
   plus Binance REST polling for OHLCV means alerts lag the market by
   low-single-digit minutes, not seconds. Fine for swing/momentum holds
   measured in hours; not a scalping tool. A truly sub-minute version needs
   the Binance WebSocket streams, not this polling loop (same bottleneck
   flagged in the v3 notebook's production roadmap).
4. **No cost/slippage/execution modeling here at all** — this is pure
   alerting. Whatever you actually pay in spread and fees when you act on an
   alert is on you to track separately.
5. **factor_weights are carried over unweighted by the robustness testing
   done in `crypto_ranked_asset_allocation_v3_robustness_testing`.** Before
   trusting this composite score's `ENTRY` alerts with real size, go back to
   that notebook's verdict table and confirm which of these 8 factors
   actually passed the hard gates (permutation significance, bootstrap CI,
   walk-forward sign consistency) — this script doesn't re-run that check on
   its own.
